# 流形筛选体系 — 实验总结与结论

## 概述

基于**流形假设**（相似光谱 → 相似CN增强特性），构建了四条互补的图流形分析路径：

| 方法 | 核心机制 | 计算复杂度 | 输出 |
|------|----------|-----------|------|
| **Label Spreading** | 已知CN标签沿KNN图软传播 | O(N²·K) | 每颗星CN概率 + z-score |
| **Spectral Clustering** | 图拉普拉斯特征分解发现CN子群 | O(N²) | 富集成簇 + 候选体 |
| **GNN (GCN/GAT)** | 端到端图神经网络节点分类 | O(N·E·d) | 每颗星CN概率 |
| **UMAP + KDE** | 流形降维 + 密度导向候选筛选 | O(N·log N) | 密度高分候选体 |

所有方法共享同一张 KNN 图 (K=50, cosine距离, 700-D 归一化光谱)，确保结果可对比。

**数据规模**: 33,565 颗恒星 (73 已知 CN, 33,492 未标记) | 图边数: 3,237,140

## 1. 环境配置

In [ ]:
import sys, os, warnings
from pathlib import Path

_PROJ = Path(os.getcwd())
for _ in range(5):
    if (_PROJ / "LabelSpreading").exists() and (_PROJ / "Data").exists():
        break
    _PROJ = _PROJ.parent
if str(_PROJ) not in sys.path:
    sys.path.insert(0, str(_PROJ))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({'font.size': 10})
import seaborn as sns; sns.set_style('whitegrid')
warnings.filterwarnings('ignore')

from data_loader import ensure_cache

data = ensure_cache()
stars_clean = data['stars_clean']
common_wave = data['common_wave']

known_mask = stars_clean['label'].values == 1
unl_mask = stars_clean['label'].values == -1
N = len(stars_clean)
n_known = known_mask.sum()
n_unl = unl_mask.sum()

print(f'数据: {N:,} 恒星, {n_known} 已知CN, {n_unl:,} 未标记 (1:{n_unl/n_known:.0f})')

## 2. 实验结果总览

In [ ]:
# Load all results
import json

OUT = _PROJ / 'LabelSpreading'

# Label Spreading
ls_probs = pd.read_csv(OUT / 'LS_all_probs.csv')
ls_cands = pd.read_csv(OUT / 'LS_candidates.csv')
ls_known_z = ls_probs[ls_probs['label'] == 1]['ls_zscore'].values
ls_unl_z = ls_probs[ls_probs['label'] == -1]['ls_zscore'].values
ls_unl_high_z = (ls_probs['label'] == -1) & (ls_probs['ls_zscore'] > 2.0)

# UMAP
umap_cands = pd.read_csv(OUT / 'UMAP_candidates.csv')

# Build summary table
summary = pd.DataFrame({
    'Method': ['Label Spreading', 'Spectral Clustering', 'GCN', 'GAT', 'UMAP+KDE'],
    'Paradigm': ['Label Propagation', 'Eigen-decomposition', 'Graph Neural Net', 'Graph Neural Net', 'Manifold Density'],
    'Precision@50': [1.00, '—', 0.34, 0.26, '—'],
    'Precision@100': [0.73, '—', 0.25, 0.19, '—'],
    '# Candidates': [
        ls_unl_high_z.sum(),
        'enriched clusters',
        '~3500 (z>2)',
        '~3500 (z>2)',
        len(umap_cands),
    ],
    'Strength': [
        'Highest P@K, conservative',
        'Discovers CN subgroups',
        'Learnable, high PR-AUC',
        'Attention over edges',
        'Complementary candidates',
    ],
    'Weakness': [
        'Low recall (few >z=2)',
        'Unsupervised, needs interpretation',
        'Overconfident, GPU heavy',
        'Slower, lower P@K',
        'No precision@K metric',
    ],
})

display(summary.style.set_caption('Four Manifold Methods — Summary'))

## 3. 方法对比可视化

In [ ]:
from IPython.display import Image, display

print('=== Label Spreading: 策略对比 ===')
display(Image(filename=str(OUT / 'strategy_comparison.png')))
print('\n=== GNN: GCN vs GAT vs LabelSpreading ===')
display(Image(filename=str(OUT / 'method_comparison.png')))
print('\n=== UMAP vs LS 候选体重叠 ===')
display(Image(filename=str(OUT / 'umap_vs_ls_overlap.png')))

## 4. Label Spreading 详细结果

### 关键指标
- **最佳策略**: spectra_cosine_K50 (700-D 光谱, cosine 距离, K=50)
- **Precision@50**: 1.00 (前50个最高概率样本全是已知CN星)
- **Precision@100**: 0.73 (前100个中73个是已知CN)
- **已知CN中位概率**: 0.53 vs 全样本中位概率 ~0.00 → 信号-背景比 > 100
- **未标记高置信候选 (z>2.0)**: 6 颗 → 极保守，极低假阳性率

### 解读
Label Spreading 的图传播在 73 颗已知 CN 星的引导下，仅在局部流形邻域产生高置信度。**73/73 已知 CN 星 z > 2.0**（100% 召回），但仅 6 颗未标记星超过此阈值——方法极其保守，几乎不产生假阳性。适合作为**高纯度量规**，而非广谱筛选。

In [ ]:
print('=== Label Spreading 候选体光谱 ===')
display(Image(filename=str(OUT / 'unlabeled_candidate_spectra.png')))
print('\n=== 已知CN vs 候选体对比 ===')
display(Image(filename=str(OUT / 'known_vs_candidate_spectra.png')))
print('\n=== 未标记候选体参数分布 ===')
display(Image(filename=str(OUT / 'unlabeled_param_space.png')))
print('\n=== K 敏感度分析 ===')
display(Image(filename=str(OUT / 'k_sensitivity.png')))

## 5. Spectral Clustering 详细结果

### 关键发现
- **特征值间隙**: 最大 eigengap 决定了自然聚类数
- **富集簇**: 存在 CN 星富集度 > 3× 全局比例的簇（全局比例 73/33565 = 0.22%）
- **候选来源**: 高富集簇中的未标记星是重点候选体

### 解读
谱聚类在**图拉普拉斯特征向量空间**中发现了 CN 星聚集的子结构。与 Label Spreading 互补——LS 沿已有边传播，谱聚类直接寻找全局图结构中的聚类。

In [ ]:
print('=== 特征值分析 ===')
display(Image(filename=str(OUT / 'eigenvalues.png')))
print('\n=== 谱嵌入空间 ===')
display(Image(filename=str(OUT / 'spectral_embedding.png')))
print('\n=== 簇富集度分析 ===')
display(Image(filename=str(OUT / 'cluster_enrichment.png')))
print('\n=== 高富集簇候选光谱 ===')
display(Image(filename=str(OUT / 'enriched_cluster_spectra.png')))

## 6. GNN (GCN + GAT) 详细结果

### 关键指标

| 模型 | Val PR-AUC | Precision@50 | Precision@100 | 参数量 |
|------|-----------|-------------|---------------|--------|
| GCN | 0.913 | 0.34 | 0.25 | 2,177 |
| GAT | 0.789 | 0.26 | 0.19 | 4,385 |

### 解读
- **GCN 优于 GAT**: 较简单的 GCN 在验证 PR-AUC 和 Precision@K 上均优于 GAT。可能原因：GAT 的注意力机制需要更多训练样本，而当前仅 43 个正训练样本
- **过自信问题**: 已知 CN 中位概率 ~0.999，但 Precision@100 仅 0.25——GNN 对已知正样本过度自信，对未标记样本的排序能力弱于 Label Spreading
- **与 LS 对比**: Label Spreading P@50=1.00 / P@100=0.73 远优于 GNN P@50=0.34 / P@100=0.25——在半监督小样本场景下，transductive 标签传播比 inductive GNN 更有效
- **3580 个 z>2 候选体**: GNN 输出的 z-score 分布更宽，识别出 3580 个未标记高 z 候选（vs LS 仅 6 个），召回率更高但精度更低

In [ ]:
print('=== GNN 训练曲线 (GCN vs GAT) ===')
display(Image(filename=str(OUT / 'training_curves.png')))
print('\n=== GNN 候选体 ===')
display(Image(filename=str(OUT / 'gnn_candidates.png')))

## 7. UMAP 流形分析详细结果

### 关键指标
- **候选体数量**: 165 颗 (top 200 by CN density)
- **与 LS 重叠**: 46/200 (23%) — **高互补性**
- **LS only**: 154 颗, **UMAP only**: 119 颗

### 解读
UMAP 嵌入 + 高斯 KDE 密度估计从完全不同的角度筛选候选体——连续流形空间 vs 离散图传播。仅 23% 的重叠率说明两种方法在捕捉**不同类型的CN信号**，联合使用可覆盖更广的候选空间。

In [ ]:
print('=== UMAP 嵌入空间 ===')
display(Image(filename=str(OUT / 'umap_embedding.png')))
print('\n=== UMAP 候选体光谱 ===')
display(Image(filename=str(OUT / 'umap_candidate_spectra.png')))
print('\n=== UMAP 候选体参数分布 ===')
display(Image(filename=str(OUT / 'umap_param_distribution.png')))

## 8. 聚类方法验证

In [ ]:
print('=== Masked-Band vs Param-Only 聚类验证 ===')
display(Image(filename=str(OUT / 'clustering_validation.png')))

## 9. 四条路径综合对比

### 候选体互补性

```
                    Label Spreading (P@50=1.00, 保守)
                    /           \
         23% overlap            23% overlap
          with UMAP              with LS
         /                               \
    UMAP+KDE                          GNN (P@50=0.34, 激进)
    (density-guided)                  (node classification)
         \                               /
          Spectral Clustering (子群发现)
```

### 方法互补矩阵

| 维度 | Label Spreading | Spectral | GCN | GAT | UMAP |
|------|:---:|:---:|:---:|:---:|:---:|
| 精度 (P@K) | ★★★★★ | — | ★★☆ | ★★☆ | — |
| 召回 (候选量) | ★☆☆ | ★★★ | ★★★★ | ★★★★ | ★★★ |
| 可解释性 | ★★★★★ | ★★★★ | ★★★ | ★★★ | ★★★ |
| 计算效率 | ★★★★ | ★★★ | ★★ | ★ | ★★★ |
| 子群发现 | ★★ | ★★★★★ | ★★ | ★★ | ★★★ |
| 互补性 | 高 | 高 | 中 | 中 | 高 |

### 各方法最佳用途

1. **Label Spreading** → 高纯度量规：当你需要"几乎确定"的CN候选体时（1-2颗新发现）
2. **Spectral Clustering** → 子群发现：寻找CN星在流形上的子结构，可能揭示不同CN增强机制
3. **GNN (GCN)** → 广谱筛选：从 ~3500 个高z候选体中用其他方法交叉验证
4. **UMAP + KDE** → 互补发现：提供与图方法不同视角的候选体

**推荐策略**: Label Spreading (高精度) + UMAP (互补覆盖) + Spectral (子群发现) 三路并行，GNN 作为补充验证。

## 10. 实验结论

### 核心发现

1. **图半监督学习对CN星检测有效且可解释**
   - Label Spreading 在 73 个已知标签的条件下，Precision@50 = 1.00, Precision@100 = 0.73
   - 图传播天然适合 CN 星检测：CN 增强是连续谱特征，流形平滑假设合理

2. **原始光谱 > 工程特征**
   - 700-D normalized spectra + cosine 距离在所有 K 值下均优于 14-D 物理特征 + Euclidean 距离
   - 验证了"让数据自己说话"优于人工特征假设

3. **Masked-band 聚类验证通过**
   - 遮蔽 CN 分子带的聚类比纯参数聚类提供更好的 z-score 分离
   - 避免了 CN 强度驱动聚类的循环偏差

4. **Inductive GNN 在小样本半监督场景不如 Transductive 传播**
   - GCN (PR-AUC=0.91) 和 GAT (PR-AUC=0.79) 的验证指标良好，但 top-K 精度远低于 Label Spreading
   - 仅有 43 个训练正样本时，直接图传播比学习参数化的节点表示更有效

5. **四种流形方法高度互补**
   - LS ∩ UMAP 重叠仅 23%，LS ∩ GNN 候选也差异显著
   - 谱聚类发现的富集簇为子群分析提供新维度
   - 四种方法从不同角度"照亮"流形，联合使用比单一方法更全面

### 与已有方法 (XGBoost PU, T_physics) 的关系

- 流形方法提供**第三维度信号**：不同于物理规则和 PU Learning，图方法基于光谱整体相似度而非分子带面积或分类边界
- 高置信候选（三方法交集）是最有希望的新 CN 星
- 各方法独有候选体代表了不同检测原理下的潜在发现

### 建议后续工作

1. **交叉验证高分候选体**: 取 LS z>3.0 + UMAP top-50 + GNN z>5.0 三路交集，手动检查光谱
2. **扩展到 1200-D 光谱**: 3800-5000Å 覆盖更多分子带特征
3. **集成预测**: 将四种流形方法的输出作为特征训练元分类器
4. **自训练迭代**: 将高置信候选加入训练集，重新运行 Label Spreading + GNN

## 附录: 输出文件清单

所有输出位于 `LabelSpreading/`:

### 图片 (PNG)
| 文件 | 方法 | 内容 |
|------|------|------|
| `strategy_comparison.png` | LS | 9种策略Precision@K + 信号-背景比 |
| `unlabeled_candidate_spectra.png` | LS | Top-16未标记候选体光谱 |
| `known_vs_candidate_spectra.png` | LS | 已知CN vs 候选体并排对比 |
| `unlabeled_param_space.png` | LS | Teff-logg空间候选体分布 |
| `k_sensitivity.png` | LS | K敏感度曲线 |
| `clustering_validation.png` | LS | Masked-band vs Param-only 聚类验证 |
| `eigenvalues.png` | Spectral | 拉普拉斯特征值 + 特征间隙 |
| `spectral_embedding.png` | Spectral | 特征向量空间2D/3D投影 |
| `cluster_enrichment.png` | Spectral | 各簇CN星富集度 |
| `enriched_cluster_spectra.png` | Spectral | 高富集簇候选体光谱 |
| `training_curves.png` | GNN | GCN vs GAT 训练曲线 |
| `method_comparison.png` | GNN | GCN vs GAT vs LS 三方法对比 |
| `gnn_candidates.png` | GNN | GNN最佳模型候选体 |
| `umap_embedding.png` | UMAP | UMAP 3视图嵌入 |
| `umap_candidate_spectra.png` | UMAP | UMAP候选体光谱 |
| `umap_param_distribution.png` | UMAP | UMAP候选体参数分布 |
| `umap_vs_ls_overlap.png` | UMAP | UMAP vs LS 候选体重叠 |

### 数据 (CSV)
| 文件 | 内容 |
|------|------|
| `LS_all_probs.csv` | 全部33,565颗星LS概率 + z-score |
| `LS_candidates.csv` | LS高置信未标记候选体 (z>2.0) |
| `UMAP_candidates.csv` | UMAP密度导向候选体 (top 200) |